<a href="https://colab.research.google.com/github/noobmaster-ru/diploma/blob/main/Computational_Methods_in_Mathematical_Economics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CMME - Computational Methods in Mathematical Economics

## Наша исходная система


\begin{cases}
\LARGE
\LARGE\frac{C_{t+1}}{C_{t}} = β(1+R_{t+1} - δ) \\
\LARGE K_{t+1} = I_{t} + (1 - δ)K_t \\
\LARGE Y_t = w_tL_t + R_tK_t = AK_t^{\alpha}L_t^{(1-\alpha)} \\
\LARGE L_{t} = (\frac{w_t}{𝛗C_t})^{\frac{1}{\psi}} \\
\LARGE R_t = α\frac{Y_t}{K_t} \\
\LARGE w_t = (1-α)\frac{Y_t}{L_t} \\
\LARGE F_t = AK_t^{\alpha}L_t^{1 - \alpha} = Y_t \\
\LARGE Y_t = C_t + S_t \quad,\quad S_t = I_t, \quad \LARGE S_t = Y_t - C_t \\
\end{cases}

\begin{cases}
\LARGE F_{t} = A*K_{t}^{\alpha}*L_{t}^{1-α} \\
\LARGE K_{t+1} = F(K_{t},L_{t}) + (1-δ)K_{t} - C_{t} \\
\LARGE \frac{C_{t+1}}{C_{t}} = β (\frac{\partial F(K_{t+1},L_{t+1})}{\partial K_{t+1}}  + 1 - δ) \\
\LARGE L_{t} = (\frac{W_t}{𝛗C_t})^{\frac{1}{\psi}} \\
\LARGE F(K_{t},L_{t}) = C_{t} + I_{t}
\end{cases}

## Импорты

In [1]:
import numpy as np
import math
from matplotlib import pyplot as plt
from dataclasses import dataclass
import plotly.graph_objects as go
from scipy.optimize import minimize_scalar, fsolve, root_scalar
from plotly.subplots import make_subplots


EPS = 1e-12
INF = 1e12

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

In [2]:
# !pip uninstall -y kaleido
# !pip install kaleido==0.2.1

## Params

In [3]:
@dataclass
class Params:
    # ==================== LEGACY ====================
    beta: float = 0.95 # discount factor
    alpha: float = 0.33 # elasticity of production wrt capital
    A: float = 0.95 # TFP
    sigma: float = 1.0 # 1/elasticity of intertemporal substitution
    delta: float = 0.1 # depreciation rate
    # ==================== LEGACY ====================
    phi: float = 0.75 # φ - фактор ненависти к труду
    psi: float = 0.35 # ψ - величина, обратная к эластичности труда по Фришу


## steady_state()

In [4]:
def steady_state(p: Params) -> dict:
    R_star = 1.0 / p.beta - 1.0 + p.delta
    v = R_star / p.alpha - p.delta
    L_star = (((1.0 - p.alpha) * R_star) / (p.alpha * p.phi * v)) ** (1.0 / (p.psi + 1.0))
    K_star = ((p.alpha * p.A * L_star**(1.0 - p.alpha)) / R_star) ** (1.0 / (1.0 - p.alpha))
    C_star = v * K_star
    Y_star = p.A * K_star**p.alpha * L_star**(1.0 - p.alpha)
    I_star = p.delta * K_star
    w_star = (1.0 - p.alpha) * Y_star / L_star
    return {
        "R_star": R_star, "v": v, "K_star": K_star, "C_star": C_star,
        "L_star": L_star, "Y_star": Y_star, "I_star": I_star, "S_star": I_star, "w_star": w_star,
    }

## Функции-расчёты

###$\LARGE L_t(K_t,C_t)=
\left(
\frac{(1-\alpha)A}{\phi}\frac{K_t^\alpha}{C_t}
\right)^{\frac{1}{\psi+\alpha}}$

In [5]:
def L_from_KC(K, C, p: Params):
    if K <= 0 or C <= 0:
        return np.nan
    base = ((1.0 - p.alpha) * p.A * K**p.alpha) / (p.phi * C)
    if base <= 0 or not np.isfinite(base):
        return np.nan
    return base ** (1.0 / (p.psi + p.alpha))


###$\LARGE Y_t(K_t,C_t) = A*K_t^{\alpha}*L_t^{(1 - \alpha)} = A*K_t^{\alpha}*L_t(K_t,C_t)$

In [6]:
def Y_from_KC(K, C, p: Params):
    L = L_from_KC(K, C, p)
    if np.isnan(L):
        return np.nan
    return p.A * K**p.alpha * L**(1.0 - p.alpha)

###$\LARGE R_t(K_t,C_t) = \alpha*\frac{Y_t}{K_t}$

In [7]:
def R_from_KC(K, C, p: Params):
    Y = Y_from_KC(K, C, p)
    return p.alpha * Y / K

###$\LARGE w_t(K_t,C_t) = (1-\alpha)*\frac{Y_t}{L_t}$

In [8]:
def w_from_KC(K, C, p: Params):
    L = L_from_KC(K, C, p)
    Y = Y_from_KC(K, C, p)
    if np.isnan(L) or np.isnan(Y):
        return np.nan
    return (1.0 - p.alpha) * Y / L

###$\LARGE I_t(Y_t,C_t) = Y_t - C_t$

In [9]:
def I_from_KC(K, C, p: Params):
    Y = Y_from_KC(K, C, p)
    if np.isnan(Y):
        return np.nan
    return Y - C

###$\LARGE K_{t+1}(I_t,K_t) = I_t + (1 - \delta)K_t$

In [10]:
def K_next_from_KC(K, C, p: Params):
    I = I_from_KC(K, C, p)
    if np.isnan(I):
        return np.nan
    return I + (1.0 - p.delta) * K

##Метод Ньютона


###$\LARGE f(c) = \frac{c}{C_t} - β(1 + R_{t+1}(K_{t+1},c) - \delta) = 0$

In [11]:
def euler_residual(C_next, K_next, C_t, p: Params):
    if C_next <= 0 or K_next <= 0 or C_t <= 0:
        return np.nan
    R_next = R_from_KC(K_next, C_next, p)
    if not np.isfinite(R_next):
        return np.nan
    return C_next / C_t - p.beta * (1.0 + R_next - p.delta)

###$\LARGE f'(c)=\frac{1}{C_t}+\beta \frac{1-\alpha}{\psi+\alpha}\frac{R(K_{t+1},c)}{c}$

(вывод прописан в тексте ВКР)

In [12]:
def euler_residual_prime(C_next, K_next, C_t, p: Params):
    if C_next <= 0 or K_next <= 0 or C_t <= 0:
        return np.nan
    R_next = R_from_KC(K_next, C_next, p)
    if not np.isfinite(R_next):
        return np.nan
    # Производная по C_next (упрощённо)
    return (1.0 / C_t) + p.beta * ((1.0 - p.alpha) / (p.psi + p.alpha)) * (R_next / C_next)


###$\LARGE c^{(n+1)}=c^{(n)}- \lambda_n \frac{f(c^{(n)})}{f'(c^{(n)})}, \quad \lambda_n \in [0,1]$

для

$\LARGE \frac{C_{t+1}}{C_t}=\beta\left(1+R(K_{t+1},C_{t+1})-\delta\right), \quad K_{t+1} = const, \quad C_t = const$

In [13]:
def solve_C_next(K_next, C_t, p: Params, tol=1e-8):
    from scipy.optimize import root_scalar
    def res(x):
        return euler_residual(x, K_next, C_t, p)
    try:
        sol = root_scalar(res, bracket=[1e-8, 1e6], method='brentq', xtol=tol)
        if sol.converged:
            return sol.root
        else:
            return np.nan
    except:
        return np.nan

## simulate_and_check()

In [14]:
def simulate_and_check(K0, C0, p, max_periods=10000, tol=1e-4):
    ss = steady_state(p)
    K_ss, C_ss, L_ss = ss['K_star'], ss['C_star'], ss['L_star']

    K = [float(K0)]
    C = [float(C0)]
    L = [L_from_KC(K0, C0, p)]
    Y = [Y_from_KC(K0, C0, p)]
    R = [R_from_KC(K0, C0, p)]
    w = [w_from_KC(K0, C0, p)]
    I = [I_from_KC(K0, C0, p)]

    if any(np.isnan(v) for v in [L[0], Y[0], R[0], w[0], I[0]]):
        return False, None, None, None, None, None, None, None, 'diverged'

    if np.isnan(L[0]) or np.isnan(Y[0]):
        return False, None, None, None, None, None, None, None, 'diverged'

    # Ожидаемое направление изменения капитала
    expect_rise = (K0 < K_ss)   # если начальный капитал ниже SS → должен расти

    for t in range(1, max_periods+1):

        K_next = K_next_from_KC(K[-1], C[-1], p)
        if np.isnan(K_next) or K_next <= 0:
            return False, None, None, None, None, None, None, None, 'diverged'

        C_next = solve_C_next(K_next, C[-1], p)
        if np.isnan(C_next) or C_next <= 0:
            return False, None, None, None, None, None, None, None, 'diverged'

        L_next = L_from_KC(K_next, C_next, p)
        Y_next = Y_from_KC(K_next, C_next, p)
        R_next = R_from_KC(K_next, C_next, p)
        w_next = w_from_KC(K_next, C_next, p)
        I_next = I_from_KC(K_next, C_next, p)

        if any(np.isnan(v) for v in [L_next, Y_next, R_next, w_next, I_next]):
            return False, None, None, None, None, None, None, None, 'diverged'

        K.append(K_next)
        C.append(C_next)
        L.append(L_next)
        Y.append(Y_next)
        R.append(R_next)
        w.append(w_next)
        I.append(I_next)

        # Проверка направления капитала
        if expect_rise:
            if K[-1] - K[-2] < 0:
                return False, np.array(K), np.array(C), np.array(L), np.array(Y), np.array(R), np.array(w), np.array(I), 'K_wrong_direction'
        else:
            if K[-1] - K[-2] > 0:
                return False, np.array(K), np.array(C), np.array(L), np.array(Y), np.array(R), np.array(w), np.array(I), 'K_wrong_direction'

        # Сходимость к SS
        if (abs(K[-1] - K_ss) < tol and
            abs(C[-1] - C_ss) < tol and
            abs(L[-1] - L_ss) < tol):
            return True, np.array(K), np.array(C), np.array(L), np.array(Y), np.array(R), np.array(w), np.array(I), 'converged'

    return False, np.array(K), np.array(C), np.array(L), np.array(Y), np.array(R), np.array(w), np.array(I), 'max_periods'


## find_C0_by_shooting()

In [15]:
def find_C0_by_shooting(K0, p, c_min=None, c_max=None, max_iter=500, tol=1e-4):
    ss = steady_state(p)
    K_ss, C_ss = ss['K_star'], ss['C_star']

    if c_min is None:
        c_min = max(1e-6, C_ss * 0.05)
    if c_max is None:
        c_max = C_ss * 100.0 if K0 > K_ss else C_ss * 2.0

    for it in range(max_iter):
        C0 = (c_min + c_max) / 2.0
        success, K_arr, C_arr, L_arr, Y_arr, R_arr, w_arr, I_arr, reason = simulate_and_check(K0, C0, p, tol=tol)

        if success:
            # print(f"Найдено C0 = {C0:.6f} за {it+1} итераций")
            T = len(K_arr) - 1
            path = {'K': K_arr, 'C': C_arr, 'L': L_arr, 'Y': Y_arr, 'R': R_arr, 'w': w_arr, 'I': I_arr}
            return C0, path, T

        # Анализ причины отказа
        if reason == 'K_wrong_direction':
            # Капитал пошёл в неправильном направлении
            if K0 < K_ss:
                # ожидался рост, но упал → слишком большое C0
                c_max = C0
            else:
                # ожидалось падение, но вырос → слишком малое C0
                c_min = C0
        else:
            # reason: 'max_periods' или 'diverged' – используем конечный капитал
            if K_arr is not None and len(K_arr) > 0:
                K_end = K_arr[-1]
                if K0 < K_ss:
                    if K_end > K_ss:
                        # перескочили через SS → нужно увеличить C0
                        c_min = C0
                    else:
                        # не достигли SS → нужно уменьшить C0
                        c_max = C0
                else:  # K0 > K_ss
                    if K_end < K_ss:
                        # перескочили через SS → нужно уменьшить C0
                        c_max = C0
                    else:
                        # не достигли SS → нужно увеличить C0
                        c_min = C0
            else:
                # если нет данных, просто сужаем интервал
                c_max = C0

        if c_max - c_min < 1e-14:
            break

    print("Не удалось подобрать C0")
    return np.nan, None, None

 ## Графики

### plot_3d_plotly()

In [16]:
def plot_3d_plotly(path, ss, T, K0):
    fig = go.Figure()

    fig.add_trace(
        go.Scatter3d(
            x=path["K"],
            y=path["L"],          # <-- теперь L на оси Y (если хотите K, L, C)
            z=path["C"],          # <-- C на оси Z
            mode="lines+markers",
            name="Траектория RBC",
            line=dict(width=6, color=path["Y"], colorscale="Viridis"),
            marker=dict(size=4, color=np.arange(len(path["K"])), colorscale="Plasma"),
            hovertemplate="K=%{x:.4f}<br>L=%{y:.4f}<br>C=%{z:.4f}<extra></extra>",
        )
    )

    # Добавляем начальную точку
    fig.add_trace(
        go.Scatter3d(
            x=[path["K"][0]],
            y=[path["L"][0]],
            z=[path["C"][0]],
            mode="markers+text",
            name="Start",
            text=["INIT"],
            textposition="top center",
            textfont=dict(size=12, color="green"),
            marker=dict(size=8, color="green", symbol="circle"),
            hovertemplate="Начальная точка<br>K=%{x:.4f}<br>L=%{y:.4f}<br>C=%{z:.4f}<extra></extra>",
        )
    )

    # Steady state точка
    fig.add_trace(
        go.Scatter3d(
            x=[ss["K_star"]],
            y=[ss["L_star"]],
            z=[ss["C_star"]],
            mode="markers+text",
            name="Steady state",
            text=["SS"],
            textposition="top center",
            textfont=dict(size=12, color="red"),
            marker=dict(size=8, color="red", symbol="diamond"),
            hovertemplate="K*=%{x:.4f}<br>L*=%{y:.4f}<br>C*=%{z:.4f}<extra></extra>",
        )
    )

    title = f"3D-траектория модели RBC: (K<sub>t</sub>, L<sub>t</sub>, C<sub>t</sub>), K<sub>0</sub> = {K0:.4f}, T = {T}"

    fig.update_layout(
        title=dict(text=title, x=0.5, font=dict(size=20)),
        scene=dict(
            xaxis=dict(title="Капитал, K<sub>t</sub>", title_font=dict(size=16),
                       showbackground=True, tickfont=dict(size=11),
                       backgroundcolor="rgba(245,245,245,1)", gridcolor="lightgray"),
            yaxis=dict(title="Труд, L<sub>t</sub>", title_font=dict(size=16),
                       tickfont=dict(size=11), showbackground=True,
                       backgroundcolor="rgba(245,245,245,1)", gridcolor="lightgray"),
            zaxis=dict(title="Потребление, C<sub>t</sub>", title_font=dict(size=16),
                       tickfont=dict(size=11), showbackground=True,
                       backgroundcolor="rgba(245,245,245,1)", gridcolor="lightgray"),
            bgcolor="rgba(245,245,245,1)",
            aspectmode="manual",           # ручное управление масштабом осей
            aspectratio=dict(x=1, y=1, z=1), # одинаковый масштаб
            camera=dict(
                eye=dict(x=1.5, y=1.5, z=1.5)   # начальный угол обзора
            )
        ),
        template="plotly_white",
        width=1000,
        height=700,
        margin=dict(l=0, r=0, b=0, t=80),
        legend=dict(x=0.02, y=0.98, font=dict(size=12)),
        font=dict(size=12),
    )

    config = {
        "toImageButtonOptions": {
            "format": "png",
            "filename": f"rbc_trajectory_3d_{K0:.4f}",
            "width": 1600,
            "height": 1200,
            "scale": 3,
        }
    }

    fig.show(config=config)
    return fig

### plot_time_series_grid_plotly()

In [17]:
def plot_time_series_grid_plotly(path, ss, T=None, K0=None):
    t = np.arange(len(path["K"]))

    mapping = [
        ("K", "Капитал, <i>K</i><sub>t</sub>", "<i>K</i><sup>*</sup>"),
        ("C", "Потребление, <i>C</i><sub>t</sub>", "<i>C</i><sup>*</sup>"),
        ("L", "Труд, <i>L</i><sub>t</sub>", "<i>L</i><sup>*</sup>"),
        ("Y", "Выпуск, <i>Y</i><sub>t</sub>", "<i>Y</i><sup>*</sup>"),
    ]

    fig = make_subplots(
        rows=2,
        cols=2,
        subplot_titles=[label for _, label, _ in mapping],
    )

    positions = [(1, 1), (1, 2), (2, 1), (2, 2)]

    for (key, label, label_star), (r, c) in zip(mapping, positions):
        fig.add_trace(
            go.Scatter(
                x=t,
                y=path[key],
                mode="lines",
                name=label,
                showlegend=True,
            ),
            row=r,
            col=c,
        )

        fig.add_trace(
            go.Scatter(
                x=t,
                y=np.full_like(t, ss[f"{key}_star"], dtype=float),
                mode="lines",
                name=label_star,
                line=dict(dash="dash"),
                showlegend=True,
            ),
            row=r,
            col=c,
        )

        fig.update_xaxes(
            title_text="t",
            title_font=dict(size=16),
            tickfont=dict(size=12),
            row=r,
            col=c,
        )
        fig.update_yaxes(
            title_text=label,
            title_font=dict(size=16),
            tickfont=dict(size=12),
            row=r,
            col=c,
        )

    if K0 is not None and T is not None:
        title = f"Переходная динамика модели RBC, K<sub>0</sub> = {K0:.6f}, T = {T}"
    elif K0 is not None:
        title = f"Переходная динамика модели RBC, K<sub>0</sub> = {K0:.6f}"
    else:
        title = "Переходная динамика модели RBC"

    fig.update_layout(
        title=dict(text=title, x=0.5, font=dict(size=20)),
        height=800,
        width=1100,
        template="plotly_white",
        font=dict(size=12),
        legend=dict(
            x=1.02,
            y=1.0,
            xanchor="left",
            yanchor="top",
            font=dict(size=12),
        ),
        margin=dict(l=80, r=180, t=90, b=70),
    )

    # размер заголовков отдельных subplot
    for ann in fig.layout.annotations:
        ann.font = dict(size=16)

    config = {
        "toImageButtonOptions": {
            "format": "png",
            "filename": f"rbc_trajectory_2d_K0_{K0:.4f}_T_{T}",
            "width": 1600,
            "height": 1100,
            "scale": 3,
        }
    }

    fig.show(config=config)
    return fig

### plot_additional_time_series()

In [18]:
def plot_additional_time_series(path, ss, T=None, K0=None):
    t = np.arange(len(path["K"]))
    mapping = [
        ("R", "Проентная ставка, <i>R</i><sub>t</sub>", "<i>R</i><sup>*</sup>"),
        ("w", "Заработная плата, <i>w</i><sub>t</sub>", "<i>w</i><sup>*</sup>"),
        ("I", "Инвестиции, <i>I</i><sub>t</sub>", "<i>I</i><sup>*</sup>"),
        ("","","")
    ]

    fig = make_subplots(
        rows=2,
        cols=2,
        subplot_titles=[label for _, label, _ in mapping],
    )

    positions = [(1, 1), (1, 2), (2, 1)]  # (2,2) остаётся пустым

    for (key, label, label_star), (r, c) in zip(mapping, positions):
        fig.add_trace(
            go.Scatter(
                x=t,
                y=path[key],
                mode="lines",
                name=label,
                showlegend=True,
            ),
            row=r,
            col=c,
        )
        fig.add_trace(
            go.Scatter(
                x=t,
                y=np.full_like(t, ss[f"{key}_star"], dtype=float),
                mode="lines",
                name=label_star,
                line=dict(dash="dash"),
                showlegend=True,
            ),
            row=r,
            col=c,
        )
        fig.update_xaxes(
            title_text="t",
            title_font=dict(size=16),
            tickfont=dict(size=12),
            row=r,
            col=c,
        )
        fig.update_yaxes(
            title_text=label,
            title_font=dict(size=16),
            tickfont=dict(size=12),
            row=r,
            col=c,
        )

    # Оформление пустой ячейки (правый нижний угол)
    fig.update_xaxes(visible=False, row=2, col=2)
    fig.update_yaxes(visible=False, row=2, col=2)

    # Заголовок графика
    if K0 is not None and T is not None:
        title = f"Дополнительные переменные RBC: R, w, I (K<sub>0</sub> = {K0:.6f}, T = {T})"
    elif K0 is not None:
        title = f"Дополнительные переменные RBC: R, w, I (K<sub>0</sub> = {K0:.6f})"
    else:
        title = "Дополнительные переменные RBC: R, w, I"

    fig.update_layout(
        title=dict(text=title, x=0.5, font=dict(size=20)),
        height=800,
        width=1100,
        template="plotly_white",
        font=dict(size=12),
        legend=dict(
            x=1.02,
            y=1.0,
            xanchor="left",
            yanchor="top",
            font=dict(size=12),
        ),
        margin=dict(l=80, r=180, t=90, b=70),
    )

    # Увеличиваем размер шрифта заголовков подграфиков
    for ann in fig.layout.annotations:
        ann.font = dict(size=16)

    config = {
        "toImageButtonOptions": {
            "format": "png",
            "filename": f"rbc_trajectory_2d_additional_K0_{K0:.4f}_T_{T}",
            "width": 1600,
            "height": 1100,
            "scale": 3,
        }
    }

    fig.show(config=config)
    return fig

### plot_3d_animated()

In [19]:
def plot_3d_animated(path, ss, T, K0):
    """
    Анимированная 3D-траектория: точка движется вдоль траектории,
    линия постепенно рисуется.
    """
    K = path["K"]
    L = path["L"]
    C = path["C"]
    Y = path["Y"]

    frames = []
    for i in range(len(K)):
        frame = go.Frame(
            data=[
                # Траектория до текущего момента
                go.Scatter3d(
                    x=K[:i+1],
                    y=L[:i+1],
                    z=C[:i+1],
                    mode="lines",
                    line=dict(width=4, color='blue'),
                    name="Прогретая траектория",
                    showlegend=False,
                ),
                # Текущая точка (движущаяся)
                go.Scatter3d(
                    x=[K[i]],
                    y=[L[i]],
                    z=[C[i]],
                    mode="markers",
                    marker=dict(size=8, color="green", symbol="circle"),
                    name="Текущая позиция",
                    showlegend=False,
                )
            ],
            name=f"frame{i}"
        )
        frames.append(frame)

    # Начальный кадр (i=0) – только первая точка
    fig = go.Figure(
        data=[
            go.Scatter3d(
                x=[K[0]],
                y=[L[0]],
                z=[C[0]],
                mode="markers",
                marker=dict(size=8, color="green", symbol="circle"),
                name="Старт",
                showlegend=True,
            ),
            go.Scatter3d(
                x=[ss["K_star"]],
                y=[ss["L_star"]],
                z=[ss["C_star"]],
                mode="markers+text",
                name="Steady state",
                text=["SS"],
                textposition="top center",
                marker=dict(size=8, color="red", symbol="diamond"),
                showlegend=True,
            )
        ],
        frames=frames
    )

    # Добавляем полную траекторию (пунктиром или бледно) для ориентира
    fig.add_trace(
        go.Scatter3d(
            x=K,
            y=L,
            z=C,
            mode="lines",
            line=dict(width=2, color='gray', dash='dash'),
            name="Полная траектория",
            showlegend=True,
        )
    )

    title = f"Анимация переходной динамики RBC: K<sub>0</sub> = {K0:.4f}, T = {T}"
    fig.update_layout(
        title=dict(text=title, x=0.5, font=dict(size=20)),
        scene=dict(
            xaxis_title="Капитал, K<sub>t</sub>",
            yaxis_title="Труд, L<sub>t</sub>",
            zaxis_title="Потребление, C<sub>t</sub>",
            aspectmode="manual",
            aspectratio=dict(x=1, y=1, z=1),
            camera=dict(eye=dict(x=1.5, y=1.5, z=1.5))
        ),
        updatemenus=[dict(
            type="buttons",
            showactive=False,
            buttons=[
                dict(label="Play",
                     method="animate",
                     args=[None, dict(frame=dict(duration=50, redraw=True),
                                      fromcurrent=True)]),
                dict(label="Pause",
                     method="animate",
                     args=[[None], dict(frame=dict(duration=0, redraw=False),
                                        mode="immediate")])
            ]
        )],
        sliders=[dict(
            steps=[
                dict(method="animate",
                     args=[[f"frame{i}"],
                           dict(mode="immediate", frame=dict(duration=0, redraw=True))],
                     label=f"{i}")
                for i in range(len(K))
            ],
            transition=dict(duration=0),
            x=0.1,
            len=0.9,
            currentvalue=dict(prefix="Время: ", font=dict(size=12))
        )],
        width=1000,
        height=700,
        margin=dict(l=0, r=0, b=0, t=80),
        legend=dict(x=0.02, y=0.98)
    )

    config = {"toImageButtonOptions": {"format": "png", "filename": f"rbc_animation_{K0:.4f}"}}
    fig.show(config=config)

### plot_phase_K_C()

In [20]:
def plot_phase_K_C(results: dict, K0: float):
    fig = go.Figure()
    colors = ['blue', 'green', 'red']
    for i, res in enumerate(results):
        path = res['path']
        ss = res['ss']
        A_val = res['A']
        fig.add_trace(go.Scatter(
            x=path['K'], y=path['C'],
            mode='lines',
            name=f'Траектория, A={A_val}',
            line=dict(color=colors[i])
        ))
        fig.add_trace(go.Scatter(
            x=[ss['K_star']], y=[ss['C_star']],
            mode='markers+text',
            name=f'SS,A={A_val}',
            marker=dict(size=10, color=colors[i], symbol='star'),
            text=[f'A={A_val}'],
            textposition='top center'
        ))
    fig.update_layout(
        title=f'Фазовый портрет (K, C), начальный капитал K0 = {K0:.4f}',
        xaxis_title='Капитал K',
        yaxis_title='Потребление C',
        template='plotly_white',
        width=1200,
        height=800,
        # Соотношение осей: единица по Y равна единице по X
        yaxis=dict(scaleanchor="x", scaleratio=1)
    )
    fig.show()

### plot_phase_K_L()

In [21]:
def plot_phase_K_L(results: dict, K0: float):
    fig = go.Figure()
    colors = ['blue', 'green', 'red']
    for i, res in enumerate(results):
        path = res['path']
        ss = res['ss']
        A_val = res['A']
        fig.add_trace(go.Scatter(
            x=path['K'], y=path['L'],
            mode='lines',
            name=f'Траектория, A={A_val}',
            line=dict(color=colors[i])
        ))
        fig.add_trace(go.Scatter(
            x=[ss['K_star']], y=[ss['L_star']],
            mode='markers+text',
            name=f'SS, A={A_val}',
            marker=dict(size=10, color=colors[i], symbol='star'),
            text=[f'A={A_val}'],
            textposition='top center'
        ))
    fig.update_layout(
        title=f'Фазовый портрет (K, L), начальный капитал K0 = {K0:.4f}',
        xaxis_title='Капитал K',
        yaxis_title='Труд L',
        template='plotly_white',
        width=1200,
        height=800,
        yaxis=dict(scaleanchor="x", scaleratio=1)
    )
    fig.show()

### plot_phase_K_C_both

In [22]:
def plot_phase_K_C_both(results_left: list[dict], results_right: list[dict], K0_left: float, K0_right: float):
    """
    Строит фазовый портрет (K, C).
    Для каждого A (0.95, 1.0, 1.05) показывает:
      - левую траекторию (K0_left) пунктиром,
      - правую траекторию (K0_right) сплошной линией,
      - точку steady state.
    """
    fig = go.Figure()
    colors = ['blue', 'green', 'red']  # для A=0.95, 1.0, 1.05 соответственно

    for i, (res_l, res_r) in enumerate(zip(results_left, results_right)):
        A_val = res_l['A']
        color = colors[i % len(colors)]

        # Левая траектория (K0_left)
        path_l = res_l['path']
        fig.add_trace(go.Scatter(
            x=path_l['K'], y=path_l['C'],
            mode='lines',
            # name=f'Траектория (K₀={K0_left:.2f}), A={A_val}',
            line=dict(color=color, dash='dash'),
            legendgroup=f'A={A_val}',
            showlegend=False
        ))

        # Правая траектория (K0_right)
        path_r = res_r['path']
        fig.add_trace(go.Scatter(
            x=path_r['K'], y=path_r['C'],
            mode='lines',
            # name=f'Траектория (K₀={K0_right:.2f}), A={A_val}',
            line=dict(color=color, dash='solid'),
            legendgroup=f'A={A_val}',
            showlegend=False
        ))

        # Steady state
        ss = res_l['ss']
        fig.add_trace(go.Scatter(
            x=[ss['K_star']], y=[ss['C_star']],
            mode='markers+text',
            name=f'SS, A={A_val}',
            marker=dict(size=12, color=color, symbol='star'),
            text=[f'A={A_val}'],
            textposition='top center',
            legendgroup=f'A={A_val}'
        ))

    fig.update_layout(
        title=f'Фазовый портрет (K, C): траектории слева (K₀={K0_left}) и справа (K₀={K0_right}) от SS',
        xaxis_title='Капитал K',
        yaxis_title='Потребление C',
        template='plotly_white',
        width=1200,
        height=800,
        # xaxis=dict(range=[3.0, 4.2]),
        # yaxis=dict(range=[1, 1.5], scaleanchor="x", scaleratio=1)
    )
    fig.show()

### plot_phase_K_L_both

In [23]:
def plot_phase_K_L_both(results_left: list[dict], results_right: list[dict], K0_left: float, K0_right: float):
    """
    Строит фазовый портрет (K, L).
    Для каждого A (0.95, 1.0, 1.05) показывает:
      - левую траекторию (K0_left) пунктиром,
      - правую траекторию (K0_right) сплошной линией,
      - точку steady state.
    """
    fig = go.Figure()
    colors = ['blue', 'green', 'red']

    for i, (res_l, res_r) in enumerate(zip(results_left, results_right)):
        A_val = res_l['A']
        color = colors[i % len(colors)]

        path_l = res_l['path']
        fig.add_trace(go.Scatter(
            x=path_l['K'], y=path_l['L'],
            mode='lines',
            name=f'Траектория (K₀={K0_left:.2f}), A={A_val}',
            line=dict(color=color, dash='dash'),
            legendgroup=f'A={A_val}',
            showlegend=False
        ))

        path_r = res_r['path']
        fig.add_trace(go.Scatter(
            x=path_r['K'], y=path_r['L'],
            mode='lines',
            name=f'Траектория (K₀={K0_right:.2f}), A={A_val}',
            line=dict(color=color, dash='solid'),
            legendgroup=f'A={A_val}',
            showlegend=False
        ))

        ss = res_l['ss']
        fig.add_trace(go.Scatter(
            x=[ss['K_star']], y=[ss['L_star']],
            mode='markers+text',
            name=f'SS, A={A_val}',
            marker=dict(size=12, color=color, symbol='star'),
            text=[f'A={A_val}'],
            textposition='top center',
            legendgroup=f'A={A_val}'
        ))

    fig.update_layout(
        title=f'Фазовый портрет (K, L): траектории слева (K₀={K0_left}) и справа (K₀={K0_right}) от SS',
        xaxis_title='Капитал K',
        yaxis_title='Труд L',
        template='plotly_white',
        width=1200,
        height=800,
        # xaxis=dict(range=[3.0, 4.3]),
        # yaxis=dict(range=[1.0, 1.2])
    )
    fig.show()

## Запуск

In [24]:
p = Params()
ss = steady_state(p)
ss

{'R_star': 0.15263157894736837,
 'v': 0.3625199362041466,
 'K_star': 3.2258188530080525,
 'C_star': 1.1694236447986124,
 'L_star': 1.101749946608162,
 'Y_star': 1.4920055300994175,
 'I_star': 0.3225818853008053,
 'S_star': 0.3225818853008053,
 'w_star': 0.9073235794057484}

In [25]:
K0 = ss['K_star'] * 10.0  # начальный капитал

C0_opt, path, T = find_C0_by_shooting(K0, p)
if path is not None:
    plot_time_series_grid_plotly(path, ss, T, K0)
else:
    print("Не удалось найти траекторию.")

In [26]:
if path is not None:
    plot_additional_time_series(path, ss, T, K0)
else:
    print("Не удалось найти траекторию.")

In [27]:
if path is not None:
    plot_3d_plotly(path, ss, T, K0)
else:
    print("Не удалось найти траекторию.")

In [28]:
if path is not None:
    plot_3d_animated(path, ss, T, K0)
else:
    print("Не удалось найти траекторию.")

## Построение траекторий (C,K), (L,K)

### solve_for_A()

In [29]:
def solve_for_A(
    A_val: float,
    K0: float,
    ss: float,
    p: Params
):
    C0_opt, path, T = find_C0_by_shooting(K0, p)
    if path is None:
        print(f"Не удалось найти траекторию для A={A_val}")
        return None
    return {
        'A': A_val,
        'path': path,
        'ss': ss,
        'T': T,
        'K0': K0
    }

### eval()

In [30]:
def eval(K0_left: float, K0_right: float):
  A_values = [0.95, 1.0, 1.05]
  results_left = []
  results_right = []
  for a in A_values:
      p = Params(A=a)
      ss = steady_state(p)

      res_left = solve_for_A(
          A_val=a,
          K0=K0_left,
          ss=ss,
          p=p,
      )
      res_right = solve_for_A(
          A_val=a,
          K0=K0_right,
          ss=ss,
          p=p,
      )
      if res_left:
          results_left.append(res_left)
      if res_right:
          results_right.append(res_right)
  return results_left, results_right

In [31]:
K0_left = 1.75
K0_right = 7.5
results_left , results_right = eval(K0_left = K0_left, K0_right = K0_right)

In [32]:
plot_phase_K_C_both(
    results_left=results_left,
    results_right=results_right,
    K0_left=K0_left,
    K0_right=K0_right
)

In [33]:
plot_phase_K_L_both(
    results_left=results_left,
    results_right=results_right,
    K0_left=K0_left,
    K0_right=K0_right
)

+======